In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

In [2]:
from Forecaster_Agent import ForecasterAgent

In [3]:
load_dotenv()

True

In [4]:
@tool
def generate_financial_forecast(days: int = 60) -> str:
    """
    Runs the BudAI mathematical forecaster to predict future bank balances.

    Args:
        days (int): The number of days into the future to forecast. 
    """
    try:
        agent = ForecasterAgent()
        real_balance = agent.fetch_live_balance()
        S0, mu, sigma = agent.fetch_and_calculate_parameters(real_balance, 60)
        agent.run_cpp_simulation(S0, mu, sigma, days=days, paths=1000)

        df = pd.read_csv("all_paths.csv", header=None)
        return (
            f"Forecast for {days} days:\n"
            f"- Careless: £{df.iloc[0].iloc[-1]:.2f}\n"
            f"- Expected: £{df.iloc[1].iloc[-1]:.2f}\n"
            f"- Optimal: £{df.iloc[2].iloc[-1]:.2f}"
        )
    except Exception as e:
        return f"Error: {str(e)}"

In [5]:
llm = ChatOllama(
    model="qwen3:8b",
    validate_model_on_init=True,
    temperature=0,
    base_url="http://localhost:11434"
).bind_tools([generate_financial_forecast])

In [ ]:
generate_financial_forecast.args_shcmea.model_json_schema()

{'description': 'Runs the BudAI mathematical forecaster to predict future bank balances.\n\nArgs:\n    days (int): The number of days into the future to forecast. ',
 'properties': {'days': {'default': 60, 'title': 'Days', 'type': 'integer'}},
 'title': 'generate_financial_forecast',
 'type': 'object'}